# <center> 相机校准

### 1. 导入包

In [1]:
import random
import numpy as np
import sys
import Arm_Lib
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
import os

### 2. 定义相机校准方法

In [2]:
class Arm_Calibration:
    def __init__(self):
        # 初始化图像
        self.image = None
        self.threshold_num = 130
        # 机械臂识别位置调节
        self.xy=[90,135]
        # 创建机械臂驱动实例
        self.arm = Arm_Lib.Arm_Device()
        self.data_collect = True 
        self.use_other_cam = False 
    
    # 绘制轮廓，返回轮廓的边点
    def calibration_map(self, image,xy=None, threshold_num=130):
        '''
        放置方块区域检测函数
        :param image:输入图像
        :return:轮廓区域边点,处理后的图像
        '''
        if xy!=None: 
            self.xy=xy
        joints_init = [self.xy[0], self.xy[1], 0, 0, 90, 0] # 机械臂初始位置角度
        self.arm.Arm_serial_servo_write6_array(joints_init, 500) # 将机械臂移动到标定方框的状态
        self.image = image
        self.threshold_num = threshold_num
        # 创建边点容器
        dp = []
        h, w = self.image.shape[:2]
        contours = self.Morphological_processing()# 获取轮廓点集(坐标)
        # 遍历点集
        for i, c in enumerate(contours):
            area = cv.contourArea(c) # 计算轮廓区域的面积
            
            low_bound = None
            high_bound = None
            if self.use_other_cam:
                low_bound = h * w / 2 - 40000
            else:
                low_bound = h * w / 2
            high_bound = h * w
            
            # 设置轮廓区域范围
            if low_bound < area < high_bound:
                mm = cv.moments(c) # 计算多边形的矩
                if mm['m00'] == 0:
                    continue
                cx = mm['m10'] / mm['m00']
                cy = mm['m01'] / mm['m00']
                # 绘制轮廓区域
                cv.drawContours(self.image, contours, i, (255, 255, 0), 2)
                # 获取轮廓区域边点
                dp = np.squeeze(cv.approxPolyDP(c, 100, True))
                # 绘制中心
                if not self.data_collect:
                    cv.circle(self.image, (np.int_(cx), np.int_(cy)), 5, (0, 0, 255), -1)
        return dp, self.image

    # 形态学及去噪处理,并获取轮廓点集
    def Morphological_processing(self):
        gray = cv.cvtColor(self.image, cv.COLOR_BGR2GRAY)# 将图像转为灰度图
        gray = cv.GaussianBlur(gray, (5, 5), 1) # 使用高斯滤镜模糊图像
        _, threshold = cv.threshold(gray, self.threshold_num, 255, cv.THRESH_BINARY) # 图像二值化操作        
        kernel = np.ones((3, 3), np.uint8) # 获取不同形状的结构元素
        # 形态学开操作
        if self.use_other_cam:
            blur = cv.morphologyEx(threshold, cv.MORPH_OPEN, kernel, iterations=8)
        else:
            blur = cv.morphologyEx(threshold, cv.MORPH_OPEN, kernel, iterations=4)
        # 提取模式
        mode = cv.RETR_EXTERNAL
        # 提取方法
        method = cv.CHAIN_APPROX_NONE
        contours, _ = cv.findContours(blur, mode, method) # 层级关系 参数一：输入的二值图，参数二：提取模式，参数三：提取方法
        return contours

    # 透视变换，将检测的轮廓透视变换为(640,480)图像
    def Perspective_transform(self, dp, image):
        '''
        :param dp: 方框边点(左上,左下,右下,右上)
        :param image: 原始图像
        :return: 透视变换后图像
        '''
        if len(dp) != 4: 
            return
        upper_left = []
        lower_left = []
        lower_right = []
        upper_right = []
        # 将边缘检测的轮廓点分为左上，左下，右下，右上
        for i in range(len(dp)):
            if dp[i][0] < 320 and dp[i][1] < 240:
                upper_left=dp[i]
            if dp[i][0] < 320 and dp[i][1] > 240:
                lower_left=dp[i]
            if dp[i][0] > 320 and dp[i][1] > 240:
                lower_right=dp[i]
            if dp[i][0] > 320 and dp[i][1] < 240:
                upper_right=dp[i]
        # 原图中的四个顶点
        pts1 = np.float32([upper_left, lower_left, lower_right, upper_right])
        # 变换后的四个顶点
        pts2 = np.float32([[0, 0], [0, 480], [640, 480], [640, 0]])
        # 根据四对对应点计算透视变换。
        M = cv.getPerspectiveTransform(pts1, pts2)
        # 将透视变换应用于图像。
        Transform_img = cv.warpPerspective(image, M, (640, 480))
        return Transform_img

### 3. 定义读写XYT的方法

In [3]:
def write_XYT(wf_path, xy, thresh):
    with open(wf_path, "w") as wf:
        str1 = 'x' + '=' + str(xy[0])
        str2 = 'y' + '=' + str(xy[1])
        str3 = 'thresh' + '=' + str(thresh)
        wf_str = str1 + '\n' + str2 + '\n' + str3
        wf.write(wf_str)
        wf.flush()


def read_XYT(rf_path):
    dict = {}
    rf = open(rf_path, "r+")
    for line in rf.readlines():
        index = line.find('=')
        dict[line[:index]] = line[index + 1:]
    xy = [int(dict['x']), int(dict['y'])]
    thresh = int(dict['thresh'])
    rf.flush()
    return xy, thresh

### 4. 创建实例,初始化参数

In [4]:
# 创建相机标定实例
calibration = Arm_Calibration()
# 创建机械臂驱动实例
arm = Arm_Lib.Arm_Device()
# 初始化一些参数
num=0
# 初始化标定方框边点
dp = []
# 初始化抓取信息
msg = {}
# 初始化1,2舵机角度值
xy = [89, 136]
# 初始化二值图阈值
threshold = 138
# 初始化模式
model = "General"
# XYT参数路径
cfg_dir = "config"
XYT_cfg_dir = os.path.join(cfg_dir, "XYT_config.txt") # XYT参数保存文件
dp_cfg_dir = os.path.join(cfg_dir, "dp.bin")
XYT_path = XYT_cfg_dir
try: 
    xy, thresh = read_XYT(XYT_path)
    print("read xy is:", xy)
except Exception: 
    print("Read XYT_config Error !!!")

Read XYT_config Error !!!


### 5. 初始化机械臂位置

In [5]:
joints_0 = [xy[0], xy[1], 0, 0, 90, 30] 
arm.Arm_serial_servo_write6_array(joints_0, 1000)

### 6. 创建控件

In [6]:
button_layout      = widgets.Layout(width='320px', height='60px', align_self='center')
output = widgets.Output()
# 调整滑杆
joint1_slider      = widgets.IntSlider(description='joint1 :'   ,    value=xy[0]     , min=70 , max=110, step=1, orientation='horizontal')
joint2_slider      = widgets.IntSlider(description='joint2 :'   ,    value=xy[1]     , min=50, max=155, step=1, orientation='horizontal') # REVISE
threshold_slider   = widgets.IntSlider(description='threshold :',    value=threshold , min=0  , max=255, step=1, orientation='horizontal')

# 进入标定模式
calibration_model  = widgets.Button(description='calibration_model',  button_style='primary', layout=button_layout)
calibration_ok     = widgets.Button(description='calibration_ok',     button_style='success', layout=button_layout)
calibration_cancel = widgets.Button(description='calibration_cancel', button_style='danger', layout=button_layout)

# 退出
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)

# 添加布局
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
garbage_identify = widgets.VBox(
    [joint1_slider, joint2_slider, threshold_slider, calibration_model, calibration_ok, calibration_cancel, exit_button],
    layout=widgets.Layout(align_self='center'));
controls_box = widgets.HBox([imgbox, garbage_identify], layout=widgets.Layout(align_self='center'))

### 7. 模式切换

In [7]:
# 标定回调
def calibration_model_Callback(value):
    global model
    model = 'Calibration'
    with output: 
        print(model)
def calibration_OK_Callback(value):
    global model
    model = 'calibration_OK'
    with output: 
        print(model)
def calibration_cancel_Callback(value):
    global model
    model = 'calibration_Cancel'
    with output: 
        print(model)
def exit_button_Callback(value):
    global model
    model = 'Exit'
    with output: 
        print(model)        
calibration_model.on_click(calibration_model_Callback)
calibration_ok.on_click(calibration_OK_Callback)
calibration_cancel.on_click(calibration_cancel_Callback)
exit_button.on_click(exit_button_Callback)

### 8. 定义主程序

In [8]:
def camera():
    global model, dp, msg, HSV_name
    # 打开摄像头
    capture = cv.VideoCapture(0)
    print("Cam has FPS:", capture.get(cv.CAP_PROP_FPS))
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            # 读取相机的每一帧
            _, img = capture.read()
            # 统一图像大小
            img = cv.resize(img, (640, 480))
            xy = [joint1_slider.value, joint2_slider.value]
            if model == 'Calibration':
                _, img = calibration.calibration_map(img, xy, threshold_slider.value)
            if model == 'calibration_OK':
                try: 
                    print("Try to write dp..")
                    write_XYT(XYT_path, xy, threshold_slider.value)
                    print("Finish Write dp..")
                except Exception: 
                    print("File XYT_config Error !!!")
                dp, img = calibration.calibration_map(img, xy, threshold_slider.value)
                print("dp is", dp)
                dp.tofile(dp_cfg_dir)
                print("saved bin..")
                model="General"
            if len(dp) != 0: 
                img = calibration.Perspective_transform(dp, img)
            if model == 'calibration_Cancel':  
                dp = []
                msg= {}
                model="General"
                
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

### 9. 启动

In [9]:
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()

Cam has FPS: 30.0
Try to write dp..